# 01 — Data Audit du dataset Tetouan

## Projet : Smart City Energy Forecasting — Tetouan  
Objectif : charger le dataset brut, vérifier sa structure, contrôler la qualité des données et préparer les premières variables cibles.

## Importation des bibliothèques

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import (
    load_tetouan_data,
    audit_tetouan_data,
    dataset_quality_summary,
)

## Définition du chemin et du renommage des colonnes

In [10]:
DATA_PATH = Path('../data/raw/Tetuan City power consumption.csv')

COLUMN_MAPPING = {
    "DateTime": "datetime",
    "Temperature": "temperature",
    "Humidity": "humidity",
    "Wind Speed": "wind_speed",
    "general diffuse flows": "general_diffuse_flows",
    "diffuse flows": "diffuse_flows",
    "Zone 1 Power Consumption": "zone1_power",
    "Zone 2  Power Consumption": "zone2_power",
    "Zone 3  Power Consumption": "zone3_power"
}

## Fonction de chargement et de préparation initiale


In [11]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Tetuan City power consumption.csv"

data = load_tetouan_data(
    DATA_PATH,
    set_datetime_index=True,
    strict_frequency=True,
)

print(dataset_quality_summary(data))
display(audit_tetouan_data(data))
print(f"Dimensions du dataset : {data.shape}")
display(data.head())


{'n_rows': 52416, 'n_columns': 10, 'start_date': '2017-01-01 00:00:00', 'end_date': '2017-12-30 23:50:00', 'inferred_frequency': '10min', 'missing_total': 0, 'duplicated_datetime': 0}


,dtype,missing_count,missing_rate_pct,min,mean,50%,max,std
temperature,float64,0,0.0,3.247000,18.810024,18.780000,40.01000,5.815476
humidity,float64,0,0.0,11.340000,68.259518,69.860000,94.80000,15.551177
wind_speed,float64,0,0.0,0.050000,1.959489,0.086000,6.48300,2.348862
general_diffuse_flows,float64,0,0.0,0.004000,182.696614,5.035500,1163.00000,264.400960
diffuse_flows,float64,0,0.0,0.011000,75.028022,4.456000,936.00000,124.210949
zone1_power,float64,0,0.0,13895.696200,32344.970564,32265.920340,52204.39512,7130.562564
zone2_power,float64,0,0.0,8560.081466,21042.509082,20823.168405,37408.86076,5201.465892
zone3_power,float64,0,0.0,5935.174070,17835.406218,16415.117470,47598.32636,6622.165099
target,float64,0,0.0,13895.696200,32344.970564,32265.920340,52204.39512,7130.562564
total_load,float64,0,0.0,36785.039739,71222.885864,69788.790940,134208.14595,17143.138964


Dimensions du dataset : (52416, 10)


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load
datetime,,,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,34055.69620,70425.53544
2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,29814.68354,69320.84387
2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,29128.10127,67803.22193
2017-01-01 00:30:00,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,28228.86076,65489.23209
2017-01-01 00:40:00,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,27335.69620,63650.44627


## Fonction d'audit de la qualité des données

In [12]:
def audit_data_quality(df):
    """
    Génère un audit de qualité du dataset :
    dimensions, période, fréquence, valeurs manquantes,
    doublons temporels et statistiques descriptives.
    """
    print("=" * 60)
    print("AUDIT GLOBAL DU DATASET")
    print("=" * 60)

    print(f"Nombre d'observations : {df.shape[0]}")
    print(f"Nombre de colonnes : {df.shape[1]}")

    print(f"Début de période : {df.index.min()}")
    print(f"Fin de période : {df.index.max()}")

    inferred_freq = pd.infer_freq(df.index)
    print(f"Fréquence temporelle inférée : {inferred_freq}")

    total_missing = df.isna().sum().sum()
    print(f"Nombre total de valeurs manquantes : {total_missing}")

    duplicated_index = df.index.duplicated().sum()
    print(f"Doublons temporels dans l'index : {duplicated_index}")

    audit = pd.DataFrame({
        "Type": df.dtypes.astype(str),
        "Valeurs Manquantes": df.isna().sum(),
        "Taux de Manquants (%)": (df.isna().mean() * 100).round(2)
    })

    summary = df.describe().T
    audit = audit.join(summary[["min", "mean", "max"]], how="left")

    return audit

# Exécution de l'audit
audit_results = audit_data_quality(data)
display(audit_results)

AUDIT GLOBAL DU DATASET
Nombre d'observations : 52416
Nombre de colonnes : 10
Début de période : 2017-01-01 00:00:00
Fin de période : 2017-12-30 23:50:00
Fréquence temporelle inférée : 10min
Nombre total de valeurs manquantes : 0
Doublons temporels dans l'index : 0


,Type,Valeurs Manquantes,Taux de Manquants (%),min,mean,max
temperature,float64,0,0.0,3.247000,18.810024,40.01000
humidity,float64,0,0.0,11.340000,68.259518,94.80000
wind_speed,float64,0,0.0,0.050000,1.959489,6.48300
general_diffuse_flows,float64,0,0.0,0.004000,182.696614,1163.00000
diffuse_flows,float64,0,0.0,0.011000,75.028022,936.00000
zone1_power,float64,0,0.0,13895.696200,32344.970564,52204.39512
zone2_power,float64,0,0.0,8560.081466,21042.509082,37408.86076
zone3_power,float64,0,0.0,5935.174070,17835.406218,47598.32636
target,float64,0,0.0,13895.696200,32344.970564,52204.39512
total_load,float64,0,0.0,36785.039739,71222.885864,134208.14595
